In [ ]:
from openai import OpenAI

openai_client = OpenAI()

In [ ]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

stream = openai_client.responses.create(
    model='gpt-4o-mini',
    input=messages,
    stream=True
)

response = None
for event in stream:
    if hasattr(event, 'delta'):
        print(event.delta, end='')
    if hasattr(event, 'response'):
        response = event.response

In [ ]:
from pydantic import BaseModel, Field

In [ ]:
class ArticleSection(BaseModel):
    header: str
    text: str = Field(description="text of the section in markdown")

class ArticleResponse(BaseModel):
    title: str
    subtitle: str
    sections: list[ArticleSection]

In [ ]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

with openai_client.responses.stream(
    model="gpt-4o-mini",
    input=messages,
    text_format=ArticleResponse,
) as stream:
    response = None
    for event in stream:
        if hasattr(event, 'delta'):
            print(event.delta, end='')
        if hasattr(event, 'response'):
            response = event.response

In [ ]:
story = response.output_parsed

In [ ]:
print('# ' + story.title)
print(story.subtitle)
print()

for section in story.sections:
    print('## ' + section.header)
    print()
    print(section.text)
    print()
    print()

In [ ]:
!uv add jaxn

In [ ]:
from jaxn import JSONParserHandler, StreamingJSONParser

In [ ]:
raw_json = """
{"title":"The Misadventures of Sparkle, the Unicorn","subtitle":"A Tale of Trouble and Lessons","sections":[{"header":"Once Upon a Time","text":"In a magical land far, far away, there lived a young unicorn named Sparkle. She was known for her shimmering coat and dazzling horn, but she had one major flaw: a mischievous streak that often led to chaos."},{"header":"The Forbidden Forest","text":"One sunny morning, Sparkle overheard her friends talking about the Forbidden Forest, a place where no unicorn dared to go. They said it was filled with danger and dark magic. But Sparkle, intrigued by the thrill, decided to venture into the forest alone."},{"header":"A Wretched Encounter","text":"As she galloped deeper into the forest, Sparkle found herself surrounded by eerie shadows and ominous sounds. Suddenly, she stumbled upon a group of grumpy trolls who were not too happy to see an intruder. Instead of being scared, Sparkle decided to play a prank on them."},{"header":"The Troll Trap","text":"Using her magic, she conjured up illusions of beautiful flowers and sparkling jewels. The trolls, greedy and foolish, fell for her trick and rushed toward the illusions. However, in doing so, they knocked over a giant boulder, blocking Sparkle's way back."},{"header":"The Consequences","text":"Trapped and surrounded by trolls who were now angrier than ever, Sparkle realized her mischief had backfired. Her only hope was to escape the trolls and unblock the path, but her magic was drained from the prank."},{"header":"A Lesson Learned","text":"Just when things seemed bleak, Sparkle remembered her true friends who would come looking for her. She calmed herself, using her wit instead of magic. She negotiated with the trolls to let her go, promising never to trespass again."},{"header":"Home Safe, but Changed","text":"With the trolls distracted by a fine tale of treasures far beyond the forest, Sparkle dashed away and eventually found her way home. Though she escaped, she learned that her mischievous nature could lead her into great trouble. From that day on, Sparkle decided to be a better friend and use her magic wisely."},{"header":"The End","text":"And so, the tale of Sparkle the unicorn spread throughout the land, reminding everyone that sometimes, the thrill of adventure can come with consequences. And being true to oneself is the most magical lesson of all."}]}
""".strip()

In [ ]:
from typing import Any, Dict

class ArticleResponseHandler(JSONParserHandler):

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '':
            if field_name == 'title':
                print(f'# {value}')
            if field_name == 'subtitle':
                print(value)
        if path == '/sections' and field_name == 'header':
            print(f'\n\n## {value}\n')

    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '/sections' and field_name == 'text':
            print(chunk, end='', flush=True)

In [ ]:
handler = ArticleResponseHandler()
parser = StreamingJSONParser(handler=handler)

In [ ]:
for c in raw_json:
    parser.parse_incremental(c)

In [ ]:
def llm_structured_stream(
        user_prompt,
        output_type,
        parser_handler=JSONParserHandler(),
        instructions=None,
        model="gpt-4o-mini",
    ):
    messages = []

    if instructions:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    parser = StreamingJSONParser(handler=parser_handler)

    with openai_client.responses.stream(
        model="gpt-4o-mini",
        input=messages,
        text_format=output_type,
    ) as stream:
        response = None
        for event in stream:
            if hasattr(event, 'delta'):
                parser.parse_incremental(event.delta)
            if hasattr(event, 'response'):
                response = event.response

    return response

In [ ]:
instructions = "your task is to tell the user bad time stories"
user_prompt =  "unicorn"

result = llm_structured_stream(
    instructions=instructions,
    user_prompt=user_prompt,
    output_type=ArticleResponse,
    parser_handler=ArticleResponseHandler(),
)

In [ ]:
import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")

def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results



instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""


prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

In [ ]:
from typing import Literal

class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [ ]:
class RAGResponseHandler(JSONParserHandler):
    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '' and field_name == 'answer':
            print(chunk, end='', flush=True)

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '' and field_name == 'answer_type':
            print('\nanswer type:', value)

    def on_array_item_end(self, path: str, field_name: str, item: Dict[str, Any] = None) -> None:
        if field_name == 'followup_questions':
            print('follow up question:', item)

In [ ]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured_stream(
        instructions=instructions,
        user_prompt=prompt,
        output_type=RAGResponse,
        parser_handler=RAGResponseHandler()
    )

In [ ]:
response = rag('llm as a judge')